In [ ]:


import numpy as np 
import pandas as pd 
import matplotlib
import matplotlib.pyplot as plt
# import lightkurve as lk 
#import asteroseismology as ast
# import astropy
import os
#get_ipython().run_line_magic('matplotlib', 'widget')
import scipy
import pymc as pm

In [ ]:


# Load in two tables - one table "stars" is the star catalog, another "modes" is the table with all the mode frequencies
stars=pd.read_excel('MOTHERSHIP/lund+17-legacy-table1 (1).xlsx')
m = stars['DeltaGamma_dip'].notna()
stars=stars[m].reset_index() #only working with stars that have a dip
modes=pd.read_csv('MOTHERSHIP/lund+17_table2.csv')

['lnK'] = modes['lnK'].apply(lambda x: 6 if x=='>1.5' else float(x))
modes=modes.rename(columns={'Freq': 'fc'})




In [ ]:


def make_fold(nu, ps, period, n_stack, n_element, idx, echelle_type="single", reverse=False):
    if echelle_type=='single':
        z = np.zeros([n_stack, n_element])
        base = np.linspace(0, period, n_element, endpoint=False) 
        for istack in range(n_stack):
            z[-istack-1,:] = np.interp(base, nu[idx[istack]:idx[istack+1]], ps[idx[istack]:idx[istack+1]], period=period)
    else:
        z = np.zeros([n_stack, 2*n_element])
        base = np.linspace(0, 2*period, 2*n_element, endpoint=False) 
        for istack in range(n_stack-1):
            if reverse:
                z[-istack-1,:] = np.r_[np.interp(base[:n_element], nu[idx[istack+1]:idx[istack+2]], ps[idx[istack+1]:idx[istack+2]], period=period),
                                       np.interp(base[:n_element], nu[idx[istack]:idx[istack+1]], ps[idx[istack]:idx[istack+1]], period=period,), ]
            else:
                z[-istack-1,:] = np.r_[np.interp(base[:n_element], nu[idx[istack]:idx[istack+1]], ps[idx[istack]:idx[istack+1]], period=period),
                                       np.interp(base[:n_element], nu[idx[istack+1]:idx[istack+2]], ps[idx[istack+1]:idx[istack+2]], period=period,), ]
    return base, z

def echelle(nu, ps, Δν, f=None, fmin=None, fmax=None, echelle_type='single', plot_with='imshow'):
    '''
    Generate a (stretched) frequency echelle plot used in asteroseismology.
    
    Parameters
    ----------
    nu : 1D array-like
        Frequencies in microhertz (μHz).
    ps : 1D array-like
        Power spectrum.
    Δν : float
        Length of each stack in microhertz (μHz).
    f : 1D array-like, optional
        Stretched frequency in seconds (μHz). Defaults to unstretched frequency echelle.
    fmin : float, optional
        Minimum frequency to be plotted. Defaults to None.
    fmax : float, optional
        Maximum frequency to be plotted. Defaults to None.
    echelle_type : str, optional
        Type of echelle diagram, either 'single' or 'replicated'. Defaults to 'single'.
    plot_with : str, optional
    Plotting method, either 'imshow' or 'contour'. Defaults to 'imshow'.

    
    Returns
    -------
    z : 2D numpy.array
        Folded power spectrum.
    extent : list
        Edges of the plot in the format [left, right, bottom, top].
    x : 1D numpy.array
        x-coordinates. Only return when 'plot_with="contour"'.
    y : 1D numpy.array
        y-coodinates. Only return when 'plot_with="contour"'.

    Notes
    -----
    The function supports two types of plotting methods: 'imshow' for a fast rendering 
    suitable for interactive use, and 'contour' for a more accurate rendering suitable 
    for publication-quality plots. 'imshow' is also accurate if no stretching involved.

    Examples
    --------
    Example Usage 1:
    Using 'plt.imshow' for fast interactive plotting:
        z, ext = echelle(nu, ps, Δν, fmin=numax-4*Dnu, fmax=numax+4*Dnu)
        plt.imshow(z, extent=ext, aspect='auto', interpolation='nearest')

    Example Usage 2:
    Using 'plt.contour' for accurate, publication-quality plotting:
        z, x, y = echelle(nu, ps, Δν, fmin=numax-4*Dnu, fmax=numax+4*Dnu, plot_with='contour')
        plt.contour(x, y, z, cmap='gray_r', levels=500)

    '''

    if fmin is None: fmin=np.nanmin(nu)
    if fmax is None: fmax=np.nanmax(nu)

    if f is None: f=np.copy(nu)

    fmin = 1e-4 if fmin<Δν else fmin - (fmin % Δν)

    # trimming
    m = (nu > fmin) & (nu < fmax)
    nu, ps, f = nu[m], ps[m], f[m] 

    # find the loci (index) of turning points to define stack
    idx = np.unique(np.concatenate([[0], np.where(np.diff((f)%Δν) < 0)[0], [len(f)-2]]))

    # define plotting elements
    resolution = np.median(np.abs(np.diff(nu)))
    # number of vertical stacks
    n_stack = len(idx) - 1 
    # number of point per stack
    n_element = int(np.ceil(Δν/resolution))
    
    # make z
    base, z = make_fold(f, ps, Δν, n_stack, n_element, idx, echelle_type=echelle_type, reverse=False)

    # format output
    if plot_with=='imshow':
        extent = (0, np.max(base), np.nanmin(nu), np.nanmax(nu)) 
        return z, extent
    elif plot_with=='contour':
        x = base
        y = np.array([np.median(nu[idx[istack]:idx[istack+1]]) for istack in range(n_stack)])
        z = np.repeat(z, 2, axis=0)
        yl = y - np.diff(y, prepend=y[0])/2
        yu = y + np.diff(y, append=y[-1])/2
        y = np.sort(np.array([yl, yu]).reshape(-1))[::-1]
        return z, x, y 
    else:
        return None

In [ ]:


def echelle_plotting(star,modes,flim):
    kic = (star['KIC'])
    psd=np.load(f'MOTHERSHIP/all_SBR_data/SBR_{kic}.npy')[:,1]
    freq=np.load(f'MOTHERSHIP/all_psds_data/psds_{kic}.npy')[:,0]
    psds=np.load(f'MOTHERSHIP/all_psds_data/psds_{kic}.npy')[:,1]
    
    plt.figure()
    plt.plot(freq,psds)
    mode = modes.query(f'KIC == "{kic}" and l <= 2 and lnK >= 1.5').reset_index(drop=True) #can change to less than 2

    row_idx = (modes['KIC'] == kic)
    #mode = modes.loc[row_idx, :]

    mode_select = mode.query('l == 0').sort_values('Amp', ascending=False).iloc[0:12]

    # we sort tables based on Amp, and select the top 12 l=0 modes
    # with the highest amplitudes
    # gather their n values -> ns 
    ns = mode_select['n']

    # then we grab l=0,1 modes with n in ns
    row_idx = mode['l'].isin([0,1,2]) & mode['n'].isin(ns)
    mode = mode.loc[row_idx, :]
    
    
    
    
    
    # Echelle diagram normalization and transformation
    norm = matplotlib.colors.Normalize(
        vmin=np.percentile(psds, 10),
        vmax=np.percentile(psds, 99.5)
    )

    z, ext = echelle(freq, psds, star['Dnu'],
                     fmin=star['numax'] * 0.2,
                     fmax=star['numax'] * 1.5,
                     echelle_type='replicated')
    
    # Plotting
    plt.figure()
    plt.imshow(z, extent=ext, aspect='auto', interpolation='nearest',
               norm=norm, cmap='gray_r')

    ells = [0, 1 ,2] #include l=2
    markers = ['o', '^', 's'] #use sqaure for l=2
    # Overlay l=0 and l=1
    for ell in ells:
        m = mode['l'] == ell
        plt.scatter(mode.loc[m,'fc']%star['Dnu'], mode.loc[m,'fc'] - mode.loc[m,'fc']%star['Dnu'] + 0.5*star['Dnu'], edgecolor=f'C{ell}', facecolor='none', marker=markers[ell] )
        plt.scatter(mode.loc[m,'fc']%star['Dnu'] + star['Dnu'], mode.loc[m,'fc'] - mode.loc[m,'fc']%star['Dnu'] - 0.5*star['Dnu'], edgecolor=f'C{ell}', facecolor='none', marker=markers[ell] )

        plt.scatter(mode.loc[m,'fc']%star['Dnu']  - flim, mode.loc[m,'fc'] - mode.loc[m,'fc']%star['Dnu'] + 0.5*star['Dnu'], c=f'C{ell}', marker='|')
        plt.scatter(mode.loc[m,'fc']%star['Dnu'] + star['Dnu'] - flim, mode.loc[m,'fc'] - mode.loc[m,'fc']%star['Dnu'] - 0.5*star['Dnu'], c=f'C{ell}', marker='|')
        plt.scatter(mode.loc[m,'fc']%star['Dnu']  + flim, mode.loc[m,'fc'] - mode.loc[m,'fc']%star['Dnu'] + 0.5*star['Dnu'], c=f'C{ell}', marker='|')
        plt.scatter(mode.loc[m,'fc']%star['Dnu'] + star['Dnu'] + flim, mode.loc[m,'fc'] - mode.loc[m,'fc']%star['Dnu'] - 0.5*star['Dnu'], c=f'C{ell}', marker='|')
    plt.xlim(0, 2*star['Dnu'])
    plt.title(f'Echelle Diagram of KIC {kic}')
    plt.savefig(f'MOTHERSHIP/echelle_diagram_l2/echelle_gamma_{kic}.pdf') #save in a different location 
    
    
    
    #peaking bagging
    def gaussian(freq, numax, w, A):
        return A * np.exp(-0.5 * ((freq - numax))**2. / (w)**2.)

    def A0(f, p):
        numax, w, A, V1, V2 = p
        height = gaussian(f, numax, w, A)
        return height

    def A1(f, p):
        numax, w, A, V1, V2 = p
        height = gaussian(f, numax, w, A)*V1
        return height
    
    def A2(f, p):
        numax, w, A, V1, V2 = p
        height = gaussian(f, numax, w, A)*V2
        return height
    

    def epsilon(i, l, m):
        if l == 0:
            return 1
        if l == 1:
            if m == 0:
                return (np.cos(i))**2.
            if np.abs(m) == 1:
                return 0.5 * (np.sin(i))**2.
        if l == 2:
            if m == 0:
                return 0.25 * ((3. * (np.cos(i))**2. - 1.))**2.
            if np.abs(m) == 1:
                return (3./8.) * (np.sin(2*i))**2.
            if np.abs(m) == 2:
                return (3./8.) * np.sin(i)**4

    def lor(x, fc, h, w):
        return h / (1.0 + 4.0*(fc - x)**2./w**2.)

    def model(x, f0_, g0, h0, f1_, g1, h1,f2_,g2, h2 ,i, split): #f0_: radial mode, f1_:dipole mode
        y_fit = np.ones(len(x))
        for n in range(len(f0_)):
            y_fit += lor(x, f0_[n], h0[n], g0[n])
        for n in range(len(f1_)):
            for m in range(-1, 2, 1):
                y_fit += lor(x, f1_[n] + (m*split), h1[n] * epsilon(i, 1, np.abs(m)), g1[n])
                
        for n in range(len(f2_)):
            for m in range(-2, 3,1):
                y_fit += lor(x, f2_[n] + (m*split), h2[n] * epsilon(i, 2, np.abs(m)), g2[n])
        return y_fit

    def logGamma(x, numax, alpha, Gamma_alpha, DGamma_dip, nu_dip, W_dip):
        return alpha * np.log(x/numax) + np.log(Gamma_alpha) + (np.log(DGamma_dip) * (1 + (2*np.log(x/nu_dip)/np.log(W_dip/numax))**2.)**-1.) 

    def fit_modes_model(KIC,freq,psd,psds,mode):

        f0_ = mode.query('l==0').fc.to_numpy()
        f1_ = mode.query('l==1').fc.to_numpy()
        f2_ = mode.query('l==2').fc.to_numpy()

        fs = np.concatenate([f0_, f1_,f2_]) #add that f2
        fs -= fs.min()
        nf = fs/fs.max()
        nf_ = nf[:,None]

        mask = freq < 0
        flim1 = 2*flim
         #change to be nus in hall paper, this is the splitting frequency
        flim2=3*flim
        
        
        for t in f0_: mask[np.abs(freq-t) < flim1] = True
        for t in f1_: mask[np.abs(freq-t) < flim1] = True
        for t in f2_: mask[np.abs(freq-t) < flim2] = True

        x, y, ys = freq[mask], psd[mask], psds[mask]

        init = {}
        init['w'] = (0.25 * star['numax'])/2.355 #width of envelope
        init['A'] = np.sqrt(np.pi* np.nanmax(y) / 2) #amplitude of envelope
        init['V1'] = 1.2 #visibility for dipole modes
        init['V2'] = 0.7 #visibility of quadupole modes #unhashtag this one
        
        
        #below should read in nus again
        init['xsplit'] = flim #mode splitting, in microHz, taken from hall paper, THIS VALUE NEEDS TO CHANGE ACCORDINGLY
        numax = star['numax']

        with pm.Model():
            # f0 = pm.Normal('f0', mu = f0_, sigma = sigma0, shape=len(self.mod.n0))
            # f1 = pm.Normal('f1', mu = f1_, sigma = sigma1, shape=len(self.mod.n1))

             # Mode linewidths
    
            alpha = pm.TruncatedNormal('alpha', mu =star['alpha'], sigma = star['e_alpha'], lower=0.)
            Gamma_alpha = pm.TruncatedNormal('Gamma_alpha', mu = star['Gamma_alpha'], sigma =star['e_Gamma_alpha'], lower=0.)
            DGamma_dip = pm.TruncatedNormal('DGamma_dip', mu =star['DeltaGamma_dip'], sigma =star['e_DeltaGamma_dip'], lower=0.)


            nu_dip = pm.Normal('nu_dip', mu=60*(numax/3090) + 2984, sigma=0.1*numax )
            W_dip = pm.Normal('W_dip', mu=-141*(numax/3090) + 4637, sigma=0.1*numax )


            # scale ratio between l=0 and l=1 modes linewidth
            scale = pm.Normal('scale', mu=1., sigma=0.05) #trying to determine what this scale value is, this is just a prior
            
            scale2 = pm.Normal('scale2', mu=1., sigma=0.05)
            
            # linewidths
            g0 = pm.Deterministic('g0', np.exp(logGamma(f0_, numax, alpha, Gamma_alpha, DGamma_dip, nu_dip, W_dip)))
            g1 = pm.Deterministic('g1', scale * np.exp(logGamma(f1_, numax, alpha, Gamma_alpha, DGamma_dip, nu_dip, W_dip)) )
            g2 = pm.Deterministic('g2', scale2 * np.exp(logGamma(f2_, numax, alpha, Gamma_alpha, DGamma_dip, nu_dip, W_dip)) )
            # g0 = pm.HalfNormal('g0', sigma=5, shape=len(f0_))
            # g1 = pm.HalfNormal('g1', sigma=5, shape=len(f1_))

            # # Mode Amplitude & Height
            w = pm.Lognormal('w', mu = np.log(init['w']), sigma = 10.)
            A = pm.Lognormal('A', mu = np.log(init['A']), sigma = 1.)
            V1 = pm.Lognormal('V1', mu = np.log(init['V1']), sigma = 0.1)
            V2 = pm.Lognormal('V2', mu = np.log(init['V2']), sigma = 0.1)

            sigmaA = pm.HalfCauchy('sigmaA', beta = 1.)
            Da0 = pm.Normal('Da0', mu = 0., sigma = 1., shape=len(f0_))
            Da1 = pm.Normal('Da1', mu = 0., sigma = 1., shape=len(f1_))
            Da2 = pm.Normal('Da2', mu = 0., sigma = 1., shape=len(f2_))

            a0 = pm.Deterministic('a0', sigmaA * Da0 + A0(f0_, [star['numax'], w, A, V1, V2]))
            a1 = pm.Deterministic('a1', sigmaA * Da1 + A1(f1_, [star['numax'], w, A, V1, V2])) 
            a2 = pm.Deterministic('a2', sigmaA * Da2  + A2(f2_, [star['numax'], w, A, V1, V2]))

            h0 = pm.Deterministic('h0', 2*(a0)**2./np.pi/g0)
            h1 = pm.Deterministic('h1', 2*(a1)**2./np.pi/g1)
            h2 = pm.Deterministic('h2', 2*(a2)**2./np.pi/g2)

            # # Mode splitting
            xsplit = pm.Lognormal('xsplit', mu = np.log(init['xsplit']), sigma= 0.75) 
            cosi = pm.Uniform('cosi', 0., 1.)

            i = pm.Deterministic('i', np.arccos(cosi))
            split = pm.Deterministic('split', xsplit/np.sin(i))

            # # # Background treatment
            # phi = pm.MvNormal('phi', mu=self.phi_, chol=self.phi_cholesky, shape=len(self.phi_))
            # phi = [8488.96] #pm.Deterministic('phi')

            # # # Construct model
            y_fit = model(x, f0_, g0, h0, f1_, g1, h1, f2_, g2, h2, i, split)

            like = pm.Gamma('like', alpha=1., beta=1./y_fit, observed=y)

            idata = pm.sample(tune = 1000,
                            draws = 1000,
                            chains = 4,
                            init = 'adapt_diag',
                            # start = self.init,
                            # initvals = self.init,
                            nuts_sampler='pymc', # 'pymc', 'numpyro', 'blackjax'
                            target_accept = 0.99)
            idata.to_netcdf(f'MOTHERSHIP/parameters_gamma_dip_l2/l2_idata_{kic}')
            return idata
    modes_model=fit_modes_model(kic,freq,psd,psds,mode)
    f0_ = mode.query('l==0').fc.to_numpy()
    f1_ = mode.query('l==1').fc.to_numpy()
    f2_ = mode.query('l==2').fc.to_numpy()
    current_idata=(f'MOTHERSHIP/parameters_gamma_dip_l2/l2_idata_{kic}')
    def sum_stat(idata,f0_,f1_,f2_):
        sum = pm.summary(idata)

        g0 = sum.loc[[t for t in sum.index if t.startswith('g0')], 'mean']
        a0 = sum.loc[[t for t in sum.index if t.startswith('a0')], 'mean']
        g1 = sum.loc[[t for t in sum.index if t.startswith('g1')], 'mean']
        a1 = sum.loc[[t for t in sum.index if t.startswith('a1')], 'mean']
        
        g2 = sum.loc[[t for t in sum.index if t.startswith('g2')], 'mean']
        a2 = sum.loc[[t for t in sum.index if t.startswith('a2')], 'mean']
        
        i = sum.loc['i', 'mean']
        split = sum.loc['split', 'mean']

        fig = plt.figure(figsize=(10, 4))
        ax = fig.gca()

        m = (freq > (f0_.min()-2*star['Dnu'])) & (freq < (f0_.max()+2*star['Dnu']))
        x, y, ys = freq[m], psd[m], psds[m]

        ax.plot(x, y, label='Power')
        # ax.plot(x, ys,)
        ax.plot(x, model(x, f0_, g0, a0, f1_, g1, a1, f2_,g2,h2, i, split), label=f'Fitted power of {kic}')
        ax.legend()
        ax.set_xlabel('ν [μHz]'); ax.set_ylabel('Power / Background');
        plt.figure()
        plt.plot(f0_, g0, 'o', label=r'$\ell=0$')
        plt.plot(f1_, g1, '^', label=r'$\ell=1$')
        plt.plot(f2_,g2, 's', label=r'$\ell=2$')
        plt.ylim(0, 6)
        plt.xlabel('ν [μHz]'); plt.ylabel('Γ [μHz]'); plt.legend()
        plt.savefig(f'MOTHERSHIP/mode_diagram_l2/mode_gamma_{kic}.pdf')
        
        
        
        
        
        #mask_modes2=modes2['l']==0
        #plt.scatter(modes2.loc[mask_modes2,'fc'],modes2.loc[mask_modes2,'Width'],color='red')
        #modify cause wont work for star1
        return
    sum_stat(current_idata,f0_,f1_,f2_)
    return

In [ ]:


hall=pd.read_csv('MOTHERSHIP/hall+21-seismic-gyro-Copy1.csv')
for istar, star in stars.loc[0:0, :].iterrows(): 
    kic=int(star['KIC'])
    match=hall[hall['KIC']==kic]
    flim=match.iloc[0]['nus*']
    echelle_plotting(star,modes,flim)

In [ ]:


stars